In [104]:
import os

In [105]:
%pwd

'C:\\Project\\CHICKEN-DISEASE-CLASSIFICATION-PROJECT'

In [106]:
os.chdir("../")

In [107]:
%pwd

'C:\\Project'

## for config entity for data classes

In [108]:
from pathlib import Path
from dataclasses import dataclass
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_weights: str
    params_classes: int
    params_include_top: bool

configuratin file


In [ ]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
        
    def get_prepare_model_config(self) -> PrepareBaseModelConfig:
        config =  self.config.prepare_base_model
        
        create_directories([config.root_dir])
        
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES,
            params_include_top=self.params.INCLUDE_TOP
        )
        
        return prepare_base_model_config
    
        
        
        

components

In [110]:
import os
import urllib.request as request
import py7zr
import tensorflow as tf

In [111]:
import os

os.chdir(r"C:\Project\CHICKEN-DISEASE-CLASSIFICATION-PROJECT")

print(os.getcwd())
print(os.path.exists(r"config\config.yaml"))

C:\Project\CHICKEN-DISEASE-CLASSIFICATION-PROJECT
True


In [112]:
import tensorflow as tf
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(
            path=self.config.base_model_path,
            model=self.model
        )

    @staticmethod
    def _prepare_full_model(
        model, classes, freeze_all, freeze_till, learning_rate
    ):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False

        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)

        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(
                learning_rate=learning_rate
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)


#for pipeline

In [113]:
try:
    config= ConfigurationManager()
    prepare_base_model_config= config.get_prepare_model_config()
    prepare_base_model=PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

Path: config\config.yaml
Content: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/itxhamzamaqsood/ml-project-image/raw/refs/heads/main/Chicken-fecal-images.7z', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_path': 'artifacts/data_ingestion'}, 'prepare_base_model': {'root_dir': 'artifacts/prepare_base_model', 'base_model_path': 'artifacts/prepare_base_model/base_model.h5', 'updated_base_model_path': 'artifacts/prepare_base_model/base_model_updatd.h5'}}
Type: <class 'dict'>
[2026-10-18 17:03:37,222: INFO: common: yaml file config\config.yaml loaded sucessfully]
Path: params.yaml
Content: {'IMAGE_SIZE': [224, 224, 3], 'BATCH_SIZE': 16, 'INCLUDE_TOP': False, 'EPOCHS': 1, 'CLASSES': 2, 'WEIGHTS': 'imagenet', 'LEARNING_RATE': 0.01, 'AUGMENTATION': True}
Type: <class 'dict'>
[2026-10-18 17:03:37,230: INFO: common: yaml file params.yaml loaded sucessfully]
[2026-10-18 17:03:37,233: INFO: common: creat

NameError: name 'get_prepare_model_config' is not defined